# EU AI Act — Citation Graph
### Step 1: Build a real graph from metadata we already have

No new tools, no new concepts yet. Just `networkx` and what's already sitting in `all_chunks.json`.

Every article chunk has a `referenced_articles` field — a list of other articles it explicitly mentions.  
That's an edge. Article 65 mentions Article 66 → `65 → 66`. Do that for all 113 articles and you have a citation graph.

What we'll discover:
- Which articles are referenced by everyone (hubs)
- Which articles reference no one and are referenced by no one (isolated)
- Which articles naturally cluster together without us defining chapters
- PageRank scores — a ranking of importance by citation structure alone
- Shortest path between any two articles through the citation network

In [1]:
import subprocess, sys

for pkg in ["networkx", "matplotlib", "python-louvain"]:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q", "--break-system-packages"],
        capture_output=True
    )

print("ready")

ready


---
## 1. Load the chunks and understand what we're working with

In [ ]:
import json

with open("./chunker_output/all_chunks.json") as f:
    all_chunks = json.load(f)

print(f"Total chunks: {len(all_chunks)}")
print()

# Separate by type
article_chunks = [c for c in all_chunks if c["metadata"]["content_type"] == "article"]
recital_chunks = [c for c in all_chunks if c["metadata"]["content_type"] == "recital"]
annex_chunks   = [c for c in all_chunks if c["metadata"]["content_type"] == "annex"]

print(f"  Article chunks : {len(article_chunks)}")
print(f"  Recital chunks : {len(recital_chunks)}")
print(f"  Annex chunks   : {len(annex_chunks)}")

In [ ]:
# Before building any graph, let's actually look at what referenced_articles contains.
# Never assume — look at the raw data first.

from pprint import pprint

sample = article_chunks[40]  # pick something mid-document
art = sample["metadata"]["article"]

print(f"Article {art['article_num']}: {art['article_title']}")
print(f"Chapter: {art['chapter_title']}")
print(f"Section: {art['section_title']}")
print(f"Page: {art['page_num']}")
print()
print(f"referenced_articles: {art['referenced_articles']}")
print(f"referenced_annexes:  {art['referenced_annexes']}")
print()
print("Text preview:")
print(sample["page_content"][:300])

In [ ]:
# Multi-chunk articles (split because they were long) have the same referenced_articles
# on every sub-chunk — because we extracted refs from the full article text.
# For graph purposes: one NODE per article_num, not per chunk.
# So deduplicate: keep only chunk_index == 0.

articles = {}
for c in article_chunks:
    art  = c["metadata"]["article"]
    num  = art["article_num"]
    # only keep the first chunk of each article
    if num not in articles:
        articles[num] = {
            "num":      num,
            "title":    art["article_title"],
            "chapter":  art["chapter_title"],
            "section":  art["section_title"],
            "page":     art["page_num"],
            "refs":     art["referenced_articles"],
            "annex_refs": art["referenced_annexes"],
        }

print(f"Unique articles: {len(articles)}")
print()

# Quick look at the distribution of how many articles each one references
ref_counts = [len(a["refs"]) for a in articles.values()]
print(f"Articles with 0 outgoing refs : {ref_counts.count(0)}")
print(f"Articles with 1-5 refs        : {sum(1 for r in ref_counts if 1 <= r <= 5)}")
print(f"Articles with 6-10 refs       : {sum(1 for r in ref_counts if 6 <= r <= 10)}")
print(f"Articles with 10+ refs        : {sum(1 for r in ref_counts if r > 10)}")
print(f"Max refs from one article     : {max(ref_counts)}")
print()

# Which article references the most others?
most_refs = max(articles.values(), key=lambda a: len(a["refs"]))
print(f"Most outgoing refs: Article {most_refs['num']} ({most_refs['title']})")
print(f"  References: {most_refs['refs']}")

---
## 2. Build the graph

`networkx` is just Python objects. A `DiGraph` (directed graph) has nodes and directed edges.  
Node = one article. Edge A→B = "Article A references Article B".  
We'll also add metadata to each node (title, chapter, page) — networkx stores these as node attributes.

In [ ]:
import networkx as nx
import re

def parse_article_num(ref_string: str):
    """Extract integer from 'Article 30' → 30. Returns None if parsing fails."""
    m = re.search(r"Article\s+(\d+)", ref_string)
    return int(m.group(1)) if m else None

# Build directed graph
G = nx.DiGraph()

# Add all article nodes with attributes
for num, art in articles.items():
    G.add_node(
        num,
        title   = art["title"],
        chapter = art["chapter"],
        section = art["section"],
        page    = art["page"],
    )

# Add edges from referenced_articles
skipped = 0
for num, art in articles.items():
    for ref in art["refs"]:
        target = parse_article_num(ref)
        if target is None:
            skipped += 1
            continue
        if target not in G:          # ref to an article not in our set
            skipped += 1
            continue
        if num == target:            # self-reference (rare but happens)
            continue
        G.add_edge(num, target)

print("Graph built!")
print(f"  Nodes (articles): {G.number_of_nodes()}")
print(f"  Edges (citations): {G.number_of_edges()}")
print(f"  Skipped refs (out-of-range or unparseable): {skipped}")
print()
print(f"  Graph density: {nx.density(G):.4f}")
print(f"  (density = actual edges / possible edges — 0 is empty, 1 is fully connected)")
print()

# Is it weakly connected? (ignoring direction)
n_components = nx.number_weakly_connected_components(G)
print(f"  Weakly connected components: {n_components}")
print(f"  (if 1: all articles connect through some path — means no isolated islands)")

---
## 3. Degree — who talks to whom the most?

In a directed graph every node has two degree counts:
- **Out-degree**: how many other articles does this one reference? (how much does it depend on others)
- **In-degree**: how many other articles reference this one? (how important is it to others)

In-degree is the raw version of importance — before PageRank weighs it.

In [ ]:
# In-degree: most referenced articles
in_deg  = sorted(G.in_degree(),  key=lambda x: x[1], reverse=True)
out_deg = sorted(G.out_degree(), key=lambda x: x[1], reverse=True)

print("TOP 15 most referenced (in-degree):")
print(f"  {'Article':<10} {'In-deg':>7}  Title")
print("  " + "-"*65)
for num, deg in in_deg[:15]:
    title = G.nodes[num]['title'][:45]
    bar = "█" * deg
    print(f"  Art {num:<6} {deg:>5}   {title}")

print()
print("TOP 15 most referencing (out-degree — depends on many others):")
print(f"  {'Article':<10} {'Out-deg':>7}  Title")
print("  " + "-"*65)
for num, deg in out_deg[:15]:
    title = G.nodes[num]['title'][:45]
    print(f"  Art {num:<6} {deg:>5}   {title}")

print()

# Zero in-degree — nobody references them
isolated_in = [num for num, deg in G.in_degree() if deg == 0]
print(f"Articles with ZERO in-degree (nobody cites them): {len(isolated_in)}")
print(f"  → These are either entry-points or definitions. Interesting for RAG — ")
print(f"    users will ask questions answered by these, but graph expansion won't surface them.")
if isolated_in[:8]:
    for n in isolated_in[:8]:
        print(f"    Art {n}: {G.nodes[n]['title'][:50]}")

---
## 4. PageRank — importance weighted by who's citing you

In-degree counts raw citations. PageRank does something smarter:
being cited by an important article is worth more than being cited by an unimportant one.  
It's the same algorithm Google originally used for web pages.

The intuition: imagine 100 people wandering the EU AI Act citation graph randomly.  
At each article they randomly follow one of its outgoing references.  
After many steps, which articles do they end up at most?  
Those are the high-PageRank articles.

For RAG: when we retrieve a chunk from Article X, we can boost its score by the PageRank  
of Article X — because PageRank tells us how structurally central that article is.

In [ ]:
# alpha=0.85 is the standard damping factor
# (with probability 0.15 the random walker teleports to a random node)
pagerank = nx.pagerank(G, alpha=0.85)

# Sort by score
pr_sorted = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)

print("PageRank TOP 20 — most structurally important articles:")
print(f"  {'Rank':<5} {'Art':>4}  {'PR score':>9}  {'In-deg':>7}  Title")
print("  " + "-"*72)
for rank, (num, score) in enumerate(pr_sorted[:20], 1):
    in_d  = G.in_degree(num)
    title = G.nodes[num]['title'][:40]
    bar   = "█" * int(score * 2000)
    print(f"  {rank:<5} {num:>4}  {score:.5f}    {in_d:>5}    {title}")

print()
print("NOTICE: PageRank vs in-degree can diverge.")
print("An article cited only by 2 other very important articles")
print("can rank higher than one cited by 8 unimportant articles.")

# Store on nodes for later use
nx.set_node_attributes(G, pagerank, 'pagerank')

In [ ]:
# Practical application: PageRank as a retrieval score booster
# When we retrieve chunks, we can re-weight their scores like this:

# Normalize PageRank to 0-1 range for clean weighting
max_pr = max(pagerank.values())
min_pr = min(pagerank.values())

def pr_boost(article_num: int, semantic_score: float,
             alpha: float = 0.8, beta: float = 0.2) -> float:
    """
    Combine semantic retrieval score with PageRank.
    alpha: weight for semantic similarity (default 80%)
    beta:  weight for structural importance (default 20%)
    """
    pr = pagerank.get(article_num, min_pr)
    pr_norm = (pr - min_pr) / (max_pr - min_pr)  # 0-1
    return alpha * semantic_score + beta * pr_norm

# Simulate what happens to retrieval ranking for two articles
print("Simulated retrieval score boosting:")
print()
scenarios = [
    (5,   0.72, "Article 5  (high PR — prohibited practices)"),
    (88,  0.75, "Article 88 (low PR — less central article)"),
    (6,   0.68, "Article 6  (high PR — high-risk AI classification)"),
    (100, 0.71, "Article 100 (low PR)"),
]
print(f"  {'Article':<40} {'Semantic':>9} {'PR score':>9} {'Final':>9}")
print("  " + "-"*72)
for art_num, sem_score, label in scenarios:
    if art_num in pagerank:
        final = pr_boost(art_num, sem_score)
        pr_raw = pagerank[art_num]
        print(f"  {label:<40} {sem_score:>9.3f} {pr_raw:>9.5f} {final:>9.3f}")

print()
print("Article 88 had a higher semantic score but Article 5 wins after the boost.")
print("This is correct — Article 5 (prohibited practices) is architecturally central.")

---
## 5. Community Detection — let the graph tell us its own groupings

We already know the chapters (defined by the legislators).  
Community detection asks: **what groupings does the citation structure itself reveal?**

Articles that heavily cross-reference each other form communities.  
The interesting question is: do these match the official chapters, or does the graph reveal  
different groupings that legislators didn't make explicit?

We use the Louvain algorithm — it finds the partition that maximises modularity  
(how much more densely connected are the communities internally vs. externally).

In [ ]:
# Community detection runs on undirected graphs
# (we lose direction info but that's fine for clustering purposes)
UG = G.to_undirected()

try:
    import community as community_louvain
    partition = community_louvain.best_partition(UG, random_state=42)
    algorithm = "Louvain"
except ImportError:
    # Fallback: greedy modularity from networkx itself
    from networkx.algorithms.community import greedy_modularity_communities
    raw = greedy_modularity_communities(UG)
    partition = {}
    for comm_id, members in enumerate(raw):
        for node in members:
            partition[node] = comm_id
    algorithm = "Greedy Modularity"

print(f"Algorithm used: {algorithm}")
print()

# Invert partition: community_id → list of article_nums
communities = {}
for node, comm_id in partition.items():
    communities.setdefault(comm_id, []).append(node)

# Sort communities by size
communities_sorted = sorted(communities.items(), key=lambda x: len(x[1]), reverse=True)

print(f"Number of communities found: {len(communities)}")
print()
print("Communities (sorted by size):")
print("-" * 70)

for comm_id, members in communities_sorted:
    members_sorted = sorted(members)
    # What chapters are in this community?
    chapters = {}
    for m in members:
        ch = G.nodes[m].get('chapter', 'Unknown')
        chapters[ch] = chapters.get(ch, 0) + 1
    dominant_chapter = max(chapters, key=chapters.get)
    dom_pct = chapters[dominant_chapter] / len(members) * 100

    print(f"Community {comm_id} ({len(members)} articles): Articles {members_sorted[:8]}{' ...' if len(members) > 8 else ''}")
    print(f"  Dominant chapter: {dominant_chapter} ({dom_pct:.0f}%)")
    if len(chapters) > 1:
        others = {k:v for k,v in chapters.items() if k != dominant_chapter}
        for ch, cnt in sorted(others.items(), key=lambda x: -x[1]):
            print(f"  Also contains:    {ch} ({cnt} articles)")
    print()

In [ ]:
# The key question: do graph communities match official chapters?
# Let's compute the overlap precisely.

# Official chapters
official_chapters = {}
for num, art in articles.items():
    ch = art["chapter"]
    official_chapters.setdefault(ch, set()).add(num)

print("Official chapters vs. graph communities:")
print("=" * 60)
print()

# For each official chapter, find which community its articles ended up in
for chapter, art_nums in sorted(official_chapters.items()):
    comm_membership = {}
    for art_num in art_nums:
        if art_num in partition:
            c = partition[art_num]
            comm_membership[c] = comm_membership.get(c, 0) + 1

    if not comm_membership:
        continue

    dominant_comm = max(comm_membership, key=comm_membership.get)
    dom_count     = comm_membership[dominant_comm]
    purity        = dom_count / len(art_nums) * 100

    print(f"{chapter}")
    print(f"  Articles: {len(art_nums)} | Mostly in community {dominant_comm} ({purity:.0f}% purity)")
    if len(comm_membership) > 1:
        split_info = ", ".join(f"comm {c}: {n}" for c,n in sorted(comm_membership.items(), key=lambda x: -x[1]))
        print(f"  Split across: {split_info}")
    print()

print()
print("HIGH purity = graph communities match official chapters (structure confirms intent)")
print("LOW purity  = articles cross-reference across chapters (interesting for RAG!)")
print("             → these are the articles you want to fetch together even if from 'different' chapters")

---
## 6. Shortest paths — how are any two articles connected?

In a citation graph, a path from A to B means:  
"Article A cites Article X, which cites Article Y, which cites Article B."

For RAG this matters because when we retrieve Article A, we might want to  
also fetch everything along the shortest path to Article B if both are relevant.  
This is what graph expansion looks like in practice — not just 1-hop, but path-following.

In [ ]:
# Some interesting path queries on the EU AI Act graph

def show_path(G, source: int, target: int):
    try:
        path = nx.shortest_path(G, source, target)
        print(f"  Shortest path Art {source} → Art {target}: length {len(path)-1} hops")
        for i, node in enumerate(path):
            prefix = "  " + ("└─ " if i == len(path)-1 else "├─ " if i > 0 else "   ")
            title  = G.nodes[node]['title'][:50]
            print(f"  {'  '*i}{'└─' if i==len(path)-1 else '├─' if i>0 else '  '} Art {node}: {title}")
    except nx.NetworkXNoPath:
        print(f"  No directed path from Art {source} to Art {target}")
    except nx.NodeNotFound as e:
        print(f"  Node not found: {e}")

print("Shortest path queries:")
print("=" * 60)
print()

pairs = [
    (5,  43, "Prohibited AI → Conformity Assessment"),
    (6,   9, "High-Risk Classification → Risk Management"),
    (10, 17, "Data Governance → Quality Management"),
    (51, 99, "GPAI → Commission review"),
]

for src, tgt, label in pairs:
    print(f"{label}:")
    show_path(G, src, tgt)
    print()

In [ ]:
# Ego graph: the immediate neighbourhood of a single article
# This is what "1-hop graph expansion" looks like in practice.
# When we retrieve Article 5, what other articles are in its 1-hop neighbourhood?

def ego_summary(G, center: int, radius: int = 1):
    ego = nx.ego_graph(G, center, radius=radius, undirected=True)
    neighbors = [n for n in ego.nodes() if n != center]

    print(f"Ego graph of Article {center}: '{G.nodes[center]['title']}'")
    print(f"  Radius: {radius} hop(s) | Neighbourhood size: {len(neighbors)} articles")
    print()

    # Split into: articles that center references vs. articles that reference center
    center_refs  = list(G.successors(center))    # center → these
    refs_center  = list(G.predecessors(center))  # these → center

    print(f"  Articles that Article {center} cites ({len(center_refs)}):")
    for n in sorted(center_refs):
        print(f"    → Art {n}: {G.nodes[n]['title'][:50]}")

    print()
    print(f"  Articles that cite Article {center} ({len(refs_center)}):")
    for n in sorted(refs_center):
        print(f"    ← Art {n}: {G.nodes[n]['title'][:50]}")

print("=" * 60)
ego_summary(G, 6)   # High-risk AI classification
print()
print("=" * 60)
ego_summary(G, 5)   # Prohibited practices


---
## 7. Graph expansion for RAG — the actual retrieval function

Now let's build the thing that matters for our pipeline.  
Given a set of retrieved chunks, use the graph to expand to related articles.

This is what we'll plug into the retrieval pipeline in a later notebook.

In [ ]:
def graph_expand_chunks(retrieved_chunks: list,
                        G: nx.DiGraph,
                        articles: dict,
                        all_chunks: list,
                        hops: int = 1,
                        pr_threshold: float = 0.0) -> dict:
    """
    Given a list of retrieved chunks, walk the citation graph to find
    related articles we should also fetch.

    Returns a dict:
      {
        'seed_articles':    [int],   # articles in retrieved chunks
        'expanded_articles': [int],  # new articles found via graph
        'expansion_reason':  {int: str}  # why each expanded article was added
      }
    """
    # Find which articles the retrieved chunks belong to
    seed_articles = set()
    for chunk in retrieved_chunks:
        meta = chunk.get("metadata", {})
        if meta.get("content_type") == "article":
            art_num = meta["article"]["article_num"]
            seed_articles.add(art_num)

    # Walk the graph outward from seeds
    expanded    = {}   # art_num → reason
    visited     = set(seed_articles)
    frontier    = set(seed_articles)

    for hop in range(hops):
        next_frontier = set()
        for art_num in frontier:
            if art_num not in G:
                continue

            # Outgoing: articles that seed references
            for neighbor in G.successors(art_num):
                if neighbor not in visited:
                    pr = pagerank.get(neighbor, 0)
                    if pr >= pr_threshold:
                        expanded[neighbor] = f"cited by Art {art_num} (hop {hop+1})"
                        next_frontier.add(neighbor)
                        visited.add(neighbor)

            # Incoming: articles that cite seed (reverse lookup)
            for neighbor in G.predecessors(art_num):
                if neighbor not in visited:
                    pr = pagerank.get(neighbor, 0)
                    if pr >= pr_threshold:
                        expanded[neighbor] = f"cites Art {art_num} (hop {hop+1})"
                        next_frontier.add(neighbor)
                        visited.add(neighbor)

        frontier = next_frontier

    return {
        "seed_articles":     sorted(seed_articles),
        "expanded_articles": sorted(expanded.keys()),
        "expansion_reason":  expanded,
    }


# Test it with a simulated retrieval result
# Pretend we retrieved chunks from Article 16 (provider obligations)
fake_retrieved = [
    {"metadata": {"content_type": "article", "article": {"article_num": 16}}},
    {"metadata": {"content_type": "article", "article": {"article_num": 9}}},
]

result = graph_expand_chunks(fake_retrieved, G, articles, all_chunks, hops=1)

print("Graph expansion result:")
print(f"  Seed articles  : {result['seed_articles']}")
print(f"  Expanded (+{len(result['expanded_articles'])} articles):")
for art_num in sorted(result['expanded_articles']):
    reason = result['expansion_reason'][art_num]
    title  = G.nodes[art_num]['title'][:50] if art_num in G else '???'
    pr     = pagerank.get(art_num, 0)
    print(f"    Art {art_num:<4} | PR={pr:.4f} | {reason}")
    print(f"           | {title}")

In [ ]:
# The pr_threshold parameter is interesting — let's see what happens when we filter
# Only expand to high-PageRank neighbours (structurally important articles)

pr_median = sorted(pagerank.values())[len(pagerank)//2]
print(f"PageRank median: {pr_median:.5f}")
print()

result_filtered = graph_expand_chunks(
    fake_retrieved, G, articles, all_chunks,
    hops=1,
    pr_threshold=pr_median  # only expand to above-average importance articles
)

print(f"With no PR filter:    {len(result['expanded_articles'])} expansion articles")
print(f"With median PR filter: {len(result_filtered['expanded_articles'])} expansion articles")
print()
print("Filtered expansion (only high-importance neighbours):")
for art_num in sorted(result_filtered['expanded_articles']):
    pr    = pagerank[art_num]
    title = G.nodes[art_num]['title'][:55] if art_num in G else '???'
    print(f"  Art {art_num:<4} | PR={pr:.4f} | {title}")

print()
print("This is exactly what we'll use in the retrieval pipeline:")
print("  1. Vector search returns top-k chunks")
print("  2. graph_expand_chunks() finds structurally related articles")
print("  3. We fetch those extra chunks and add them to the LLM context")
print("  4. LLM sees richer, better-connected context → better answers")

---
## 8. Visualise the graph

Before saving, let's actually look at it. We'll use a spring layout —  
articles that cite each other pull toward each other, isolated ones drift outward.  
Colour by chapter, size by PageRank.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Chapter → colour mapping
chapter_list = sorted(set(data['chapter'] for _, data in G.nodes(data=True)))
cmap = plt.cm.get_cmap('tab10', len(chapter_list))
chapter_to_color = {ch: cmap(i) for i, ch in enumerate(chapter_list)}

node_colors = [chapter_to_color[G.nodes[n]['chapter']] for n in G.nodes()]

# Node size = PageRank (scaled)
pr_vals     = [pagerank.get(n, 0) for n in G.nodes()]
max_pr_val  = max(pr_vals)
node_sizes  = [200 + 2000 * (pr / max_pr_val) for pr in pr_vals]

# Compute spring layout (k controls spacing)
print("Computing spring layout... (may take a few seconds)")
pos = nx.spring_layout(G, k=1.2, iterations=80, seed=42)

fig, ax = plt.subplots(figsize=(18, 14))
ax.set_facecolor('#0f0f0f')
fig.patch.set_facecolor('#0f0f0f')

# Draw edges first (behind nodes)
nx.draw_networkx_edges(
    G, pos, ax=ax,
    edge_color='#ffffff', alpha=0.08,
    arrows=True, arrowsize=8,
    width=0.5,
    connectionstyle='arc3,rad=0.1'
)

# Draw nodes
nx.draw_networkx_nodes(
    G, pos, ax=ax,
    node_color=node_colors,
    node_size=node_sizes,
    alpha=0.9,
    linewidths=0.5,
    edgecolors='white'
)

# Label only high-PageRank nodes (top 20) to keep it readable
top_nodes = [n for n, _ in pr_sorted[:20]]
labels    = {n: f"A{n}" for n in top_nodes}
nx.draw_networkx_labels(
    G, pos, labels=labels, ax=ax,
    font_color='white', font_size=7, font_weight='bold'
)

# Legend for chapters
patches = [mpatches.Patch(color=chapter_to_color[ch], label=ch[:30])
           for ch in chapter_list]
ax.legend(handles=patches, loc='lower left', fontsize=7,
          framealpha=0.3, facecolor='#222222', labelcolor='white')

ax.set_title("EU AI Act — Article Citation Graph\n"
             "Node size = PageRank | Colour = Chapter | Labels on top-20 by PageRank",
             color='white', fontsize=13, pad=16)
ax.axis('off')

plt.tight_layout()
plt.savefig("eu_ai_act_citation_graph.png", dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("Saved: eu_ai_act_citation_graph.png")

---
## 9. Add recitals to the graph

Recitals explain articles. They're the legal commentary —  
"why did we write Article 5 this way? Read Recital 43."

Every recital chunk has `referenced_articles` too.  
Let's add them as a second type of node and see the bipartite structure.

In [ ]:
# Build recital → article edges
# Node naming: articles are ints (6, 9, 16...)
#              recitals will be strings ('R1', 'R43', ...) to avoid collision

G2 = G.copy()  # keep the article-only graph intact

recital_index = {}
for c in recital_chunks:
    rec  = c["metadata"]["recital"]
    rnum = rec["recital_num"]
    node_id = f"R{rnum}"

    if node_id in recital_index:
        continue  # deduplicate
    recital_index[node_id] = rec

    G2.add_node(node_id, kind="recital", num=rnum)

    for ref in rec.get("referenced_articles", []):
        target = parse_article_num(ref)
        if target and target in G2:
            G2.add_edge(node_id, target, edge_type="explains")

print(f"Extended graph with recitals:")
print(f"  Total nodes: {G2.number_of_nodes()} ({len(recital_index)} recitals + {G.number_of_nodes()} articles)")
print(f"  Total edges: {G2.number_of_edges()}")
print()

# Which articles have the most recitals explaining them?
recital_coverage = {}
for r_node_id, rec in recital_index.items():
    for ref in rec.get("referenced_articles", []):
        target = parse_article_num(ref)
        if target:
            recital_coverage[target] = recital_coverage.get(target, 0) + 1

top_covered = sorted(recital_coverage.items(), key=lambda x: x[1], reverse=True)[:12]
print("Articles with the most recitals explaining them:")
for art_num, count in top_covered:
    title = G.nodes[art_num]['title'][:50] if art_num in G else '???'
    print(f"  Art {art_num:<4} ← {count:>3} recitals | {title}")

print()
print("RAG implication: when you retrieve Article X, also fetch the recitals that explain it.")
print("Those recitals contain the legislative INTENT — crucial for legal interpretation questions.")

In [ ]:
# Build the lookup function we'll use in the retrieval pipeline

def get_explaining_recitals(article_num: int, G2: nx.DiGraph) -> list:
    """
    Given an article number, find all recital node IDs that reference it.
    Returns list of recital node IDs e.g. ['R43', 'R57', 'R89']
    """
    recitals = []
    for pred in G2.predecessors(article_num):
        if isinstance(pred, str) and pred.startswith('R'):
            recitals.append(pred)
    return sorted(recitals, key=lambda r: int(r[1:]))

# Test on the most-covered articles
for art_num, _ in top_covered[:5]:
    recs = get_explaining_recitals(art_num, G2)
    title = G.nodes[art_num]['title'][:45] if art_num in G else '???'
    print(f"  Art {art_num} ({title})")
    print(f"    Explained by: {recs}")
    print()

---
## 10. Save the graph

We'll save two things:
1. The full graph as a JSON (nodes + edges + all attributes) — easy to reload in any notebook
2. PageRank scores as a JSON — we'll import this into the retrieval pipeline for score boosting

In [ ]:
import json

# --- Save 1: graph as node-link JSON ---
# networkx has a built-in serializer for this
from networkx.readwrite import json_graph

graph_data = json_graph.node_link_data(G)  # article-only graph

# node IDs are ints — json serialises fine
with open("citation_graph.json", "w") as f:
    json.dump(graph_data, f, indent=2)

print(f"Saved: citation_graph.json")
print(f"  Nodes: {len(graph_data['nodes'])}")
print(f"  Edges: {len(graph_data['links'])}")

# --- Save 2: PageRank scores ---
pr_export = {str(k): round(v, 8) for k, v in pagerank.items()}
with open("article_pagerank.json", "w") as f:
    json.dump(pr_export, f, indent=2)

print(f"\nSaved: article_pagerank.json")
print(f"  {len(pr_export)} articles with PR scores")

# --- Save 3: community membership ---
comm_export = {str(k): v for k, v in partition.items()}
with open("article_communities.json", "w") as f:
    json.dump(comm_export, f, indent=2)

print(f"\nSaved: article_communities.json")
print(f"  {len(communities)} communities")

# --- Save 4: recital → article mapping ---
recital_map = {}
for r_node_id, rec in recital_index.items():
    recital_map[r_node_id] = {
        "recital_num": rec["recital_num"],
        "explains_articles": [
            int(n) for n in G2.successors(r_node_id)
            if isinstance(n, int)
        ]
    }

with open("recital_article_map.json", "w") as f:
    json.dump(recital_map, f, indent=2)

print(f"\nSaved: recital_article_map.json")
print(f"  {len(recital_map)} recitals mapped to their articles")

In [ ]:
# Verify the save/load round-trip works
# This is what every future notebook will do at the top

from networkx.readwrite import json_graph
import json

with open("citation_graph.json") as f:
    data = json.load(f)

G_loaded = json_graph.node_link_graph(data)

# node IDs come back as strings from JSON — convert back to ints
G_loaded = nx.relabel_nodes(G_loaded, {n: int(n) if str(n).isdigit() else n
                                        for n in G_loaded.nodes()})

with open("article_pagerank.json") as f:
    pr_loaded = {int(k): v for k, v in json.load(f).items()}

with open("article_communities.json") as f:
    comm_loaded = {int(k): v for k, v in json.load(f).items()}

print("Round-trip check:")
print(f"  Nodes: {G_loaded.number_of_nodes()} (original: {G.number_of_nodes()})  {'✓' if G_loaded.number_of_nodes() == G.number_of_nodes() else '✗'}")
print(f"  Edges: {G_loaded.number_of_edges()} (original: {G.number_of_edges()})  {'✓' if G_loaded.number_of_edges() == G.number_of_edges() else '✗'}")
print(f"  PR scores: {len(pr_loaded)} articles  {'✓' if len(pr_loaded) == len(pagerank) else '✗'}")
print(f"  Communities: {len(set(comm_loaded.values()))} communities  {'✓' if len(set(comm_loaded.values())) == len(communities) else '✗'}")
print()
print("All saved. Next notebook: Step 2 — graph algorithms, deeper analysis.")
print("After that: Step 3 — LLM entity extraction to build a richer graph from raw text.")

---
## What we built and what it means

**The graph:**
- Nodes = 113 articles, each with title, chapter, section, page, PageRank
- Edges = citation references extracted from chunk metadata
- Separate recital layer that connects recitals to the articles they explain

**What the algorithms told us:**
- **In-degree** → which articles are cited most (raw popularity)
- **PageRank** → which articles are *structurally* most important (weighted by who's citing them)
- **Communities** → which articles cross-reference so heavily they form natural clusters, independent of the chapter structure the legislators defined
- **Shortest paths** → the citation chains connecting any two articles through intermediate references
- **Ego graphs** → the 1-hop neighbourhood of any article = what to fetch in addition to it

**What we saved for the retrieval pipeline:**
- `citation_graph.json` — full graph, loadable in any notebook
- `article_pagerank.json` — PR scores for score boosting at retrieval time
- `article_communities.json` — community membership for query routing
- `recital_article_map.json` — which recitals explain which articles

**What's still missing (Step 3):**
This graph only uses *explicit* references (`referenced_articles` from our chunker).  
It does not know that `Notified Body` and `Provider` and `Conformity Assessment` are  
entities that connect articles in *implicit* ways — through shared concepts, not just hyperlinks.  
That's what LLM entity extraction gives us. The graph will be 10x richer.